# Verifikasi Gap Ekstrem — Artefak atau Kegagalan Model?

Temuan sebelumnya: pada beberapa metrik (terutama coherence & relevance) ada dialog yang skornya melompat
dari ~0.04 di satu config ke ~0.97 di config lain. Sebelum menyimpulkan apa pun, kita harus tahu apakah
skor sangat rendah itu:
- **artefak teknis** (output kosong/terpotong, repetisi degeneratif, salah alignment antar file), atau
- **kegagalan model sungguhan** (output wajar tapi memang jelek).

Alur:
1. Muat skor + teks output tiap config (paired).
2. Tandai dialog dengan skor sangat rendah / gap sangat besar.
3. Hitung indikator artefak otomatis (panjang output, kosong, repetisi).
4. Cetak teks dialog tersangka untuk inspeksi manual.
5. Ekspor daftar tersangka + verdict awal.

## 1. Muat skor & teks (auto-deteksi field)

Sel ini mencetak kunci yang tersedia di file teks tiap config. Jika tebakan field teks salah, ganti
`TEXT_KEYS_CANDIDATES` atau set `TEXT_KEY` manual.

In [1]:
import json
from pathlib import Path
import numpy as np, pandas as pd

def load_json(path):
    p = Path(path)
    if not p.is_file():
        raise FileNotFoundError(f"Tidak ditemukan: {p}")
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

# ---- sesuaikan bila perlu ----
METHOD = "method4"
CONFIGS = {
    "margin_part": f"{METHOD}_outputs/bart_margin_part",
    "margin_full": f"{METHOD}_outputs/bart_margin_full",
    "finetune":    f"{METHOD}_outputs/bart_finetune",
}
TEXT_FILE = "generated.json"          # nama file teks output per config (ganti bila beda)
LING = ["coherence", "relevance", "fluency", "consistency", "alignscore"]
# kandidat nama field untuk teks output model & input/konteks (urutan = prioritas tebak)
TEXT_KEYS_CANDIDATES   = ["prediction", "generated", "output", "hypothesis", "response", "pred", "summary", "text"]
SOURCE_KEYS_CANDIDATES = ["source", "input", "context", "dialogue", "dialog", "document", "src"]
REF_KEYS_CANDIDATES    = ["reference", "target", "gold", "ref", "label"]
TEXT_KEY = None  # set manual untuk override, mis. TEXT_KEY = "prediction"
# --------------------------------

def load_unieval(folder):
    data = load_json(f"{folder}/unieval.json")
    try:
        align = load_json(f"{folder}/alignscore.json")
        data = [{**r, "alignscore": s} for r, s in zip(data, align)]
    except FileNotFoundError:
        pass
    return pd.DataFrame(data)

def pick_key(sample_dict, candidates):
    for c in candidates:
        if c in sample_dict:
            return c
    return None

scores = {name: load_unieval(folder) for name, folder in CONFIGS.items()}
texts_raw = {name: load_json(f"{folder}/{TEXT_FILE}") for name, folder in CONFIGS.items()}

# tampilkan kunci yang ada di file teks
print("="*64); print("KUNCI DI FILE TEKS tiap config"); print("="*64)
for name, arr in texts_raw.items():
    ks = list(arr[0].keys()) if (isinstance(arr, list) and arr and isinstance(arr[0], dict)) else type(arr[0]).__name__
    print(f"  {name:12s} n={len(arr):4d}  keys={ks}")

sample = texts_raw[list(CONFIGS)[0]][0]
if isinstance(sample, dict):
    text_key = TEXT_KEY or pick_key(sample, TEXT_KEYS_CANDIDATES)
    src_key  = pick_key(sample, SOURCE_KEYS_CANDIDATES)
    ref_key  = pick_key(sample, REF_KEYS_CANDIDATES)
else:
    text_key = src_key = ref_key = None  # file teks adalah list of string
print(f"\nDipakai -> text_key={text_key!r}  source_key={src_key!r}  ref_key={ref_key!r}")
if text_key is None and not isinstance(sample, dict):
    print("(File teks tampaknya list of string; teks diambil langsung sebagai elemen.)")
elif text_key is None:
    print("!! Tidak menemukan field teks. Set TEXT_KEY manual lalu jalankan ulang.")

KUNCI DI FILE TEKS tiap config
  margin_part  n= 499  keys=['prompt', 'prediction', 'reference']
  margin_full  n= 499  keys=['prompt', 'prediction', 'reference']
  finetune     n= 499  keys=['prompt', 'prediction', 'reference']

Dipakai -> text_key='prediction'  source_key=None  ref_key='reference'


## 2. Samakan panjang (paired) & rakit teks

Memastikan skor & teks tiap config punya jumlah dialog yang sama dan terindeks identik.

In [2]:
def get_text(arr, i, key):
    item = arr[i]
    if isinstance(item, dict):
        return str(item.get(key, ""))
    return str(item)

n = min(min(len(scores[k]) for k in scores), min(len(texts_raw[k]) for k in texts_raw))
print("n paired =", n)

txt = {}
for k in CONFIGS:
    txt[k] = [get_text(texts_raw[k], i, text_key) for i in range(n)]
    scores[k] = scores[k].iloc[:n].reset_index(drop=True)

# matriks skor per metrik (dialog x config)
M = {m: pd.DataFrame({k: scores[k][m].values[:n] for k in CONFIGS}) for m in LING}
for m in LING: M[m].index.name = "dialog_id"

n paired = 499


## 3. Tandai dialog tersangka

Dua kriteria:
- **skor rendah absolut**: skor < `LOW_THR` pada metrik mana pun di config mana pun.
- **gap besar**: selisih (max−min) lintas config ≥ `GAP_THR` pada suatu metrik.

Untuk tiap (dialog, config) yang bermasalah, kita hitung indikator artefak dari teksnya.

In [3]:
import re
LOW_THR = 0.10   # skor di bawah ini dianggap 'sangat rendah'
GAP_THR = 0.60   # gap antar config di atas ini dianggap 'ekstrem'

def artifact_flags(s):
    """Indikator artefak dari satu string output."""
    s = s or ""
    toks = s.split()
    nt = len(toks)
    empty = (nt == 0)
    very_short = (nt < 3)
    # repetisi: rasio token unik rendah, atau ada n-gram berulang beruntun
    uniq_ratio = (len(set(toks)) / nt) if nt else 0.0
    max_run = 0
    if nt:
        run = 1
        for a, b in zip(toks, toks[1:]):
            run = run + 1 if a == b else 1
            max_run = max(max_run, run)
    degenerate = (nt >= 5 and uniq_ratio < 0.5) or (max_run >= 4)
    # terpotong: tidak diakhiri tanda baca kalimat
    truncated = (nt >= 3 and not re.search(r"[.!?\u3002]$", s.strip()))
    return dict(n_tokens=nt, empty=empty, very_short=very_short,
                uniq_ratio=round(uniq_ratio, 2), max_run=max_run,
                degenerate=degenerate, truncated=truncated)

rows = []
for i in range(n):
    for m in LING:
        vals = {k: M[m].loc[i, k] for k in CONFIGS}
        gap = max(vals.values()) - min(vals.values())
        for k in CONFIGS:
            sc = vals[k]
            if sc < LOW_THR or gap >= GAP_THR:
                af = artifact_flags(txt[k][i])
                rows.append({"dialog_id": i, "metric": m, "config": k,
                             "score": round(sc, 4), "gap": round(gap, 4),
                             "low_score": sc < LOW_THR, **af})
flags = pd.DataFrame(rows)
print("Total baris tersangka (dialog×metrik×config):", len(flags))
if len(flags):
    print("\nRingkasan indikator artefak pada baris ber-skor sangat rendah:")
    low = flags[flags["low_score"]]
    print("  jumlah baris low-score :", len(low))
    for col in ["empty", "very_short", "degenerate", "truncated"]:
        print(f"  {col:11s}: {int(low[col].sum())} dari {len(low)}")
    print("\n  mean panjang token (low-score) :", round(low["n_tokens"].mean(), 1))

Total baris tersangka (dialog×metrik×config): 254

Ringkasan indikator artefak pada baris ber-skor sangat rendah:
  jumlah baris low-score : 34
  empty      : 0 dari 34
  very_short : 0 dari 34
  degenerate : 0 dari 34
  truncated  : 20 dari 34

  mean panjang token (low-score) : 92.3


## 4. Verdict awal per (dialog, config) ber-skor rendah

Aturan sederhana: jika ada indikator artefak → `ARTEFAK`; jika output wajar (cukup panjang, tak terpotong,
tak degeneratif) tapi skor tetap rendah → `KEGAGALAN_MODEL` (output beneran jelek). Verdict ini hanya
penyaring — tetap baca teksnya di sel 5.

In [4]:
def verdict(r):
    if r["empty"]:        return "ARTEFAK_kosong"
    if r["very_short"]:   return "ARTEFAK_terlalu_pendek"
    if r["degenerate"]:   return "ARTEFAK_repetisi"
    if r["truncated"]:    return "cek_terpotong?"
    return "KEGAGALAN_MODEL?"

low = flags[flags["low_score"]].copy()
if len(low):
    low["verdict"] = low.apply(verdict, axis=1)
    print("="*64); print("DISTRIBUSI VERDICT (baris ber-skor sangat rendah)"); print("="*64)
    print(low["verdict"].value_counts().to_string())
    print("\nPer metrik:")
    print(pd.crosstab(low["metric"], low["verdict"]).to_string())
    print("\nPer config:")
    print(pd.crosstab(low["config"], low["verdict"]).to_string())
else:
    print("Tidak ada baris ber-skor di bawah LOW_THR. Naikkan LOW_THR bila ingin memeriksa lebih banyak.")

DISTRIBUSI VERDICT (baris ber-skor sangat rendah)
verdict
cek_terpotong?      20
KEGAGALAN_MODEL?    14

Per metrik:
verdict    KEGAGALAN_MODEL?  cek_terpotong?
metric                                     
coherence                 6               8
relevance                 8              12

Per config:
verdict      KEGAGALAN_MODEL?  cek_terpotong?
config                                       
finetune                    6              11
margin_full                 4               8
margin_part                 4               1


## 5. Inspeksi manual — cetak teks dialog tersangka

Untuk tiap dialog tersangka teratas, tampilkan skor SEMUA config berdampingan + teks output tiap config
(plus source/reference bila ada). Inilah bukti untuk memutuskan artefak vs kegagalan sungguhan.

In [8]:
TOP = 8        # berapa dialog tersangka yang dicetak
MAXCHAR = 1000  # potong teks panjang biar terbaca

# pilih dialog tersangka unik, prioritaskan skor terendah
if len(flags):
    cand = (flags.sort_values("score")
                 .drop_duplicates("dialog_id")
                 .head(TOP)["dialog_id"].tolist())
    cand.append(135)
else:
    cand = []

def clip(s): return (s[:MAXCHAR] + " …[clip]") if len(s) > MAXCHAR else s

for i in cand:
    print("█"*72)
    print(f"DIALOG {i}")
    print("█"*72)
    sc_tbl = pd.DataFrame({m: {k: round(M[m].loc[i, k], 3) for k in CONFIGS} for m in LING})
    print(sc_tbl.to_string())
    first = texts_raw[list(CONFIGS)[0]][i]
    if isinstance(first, dict) and src_key:
        print(f"\n[SOURCE] {clip(str(first.get(src_key,'')))}")
    if isinstance(first, dict) and ref_key:
        print(f"[REF]    {clip(str(first.get(ref_key,'')))}")
    for k in CONFIGS:
        af = artifact_flags(txt[k][i])
        tag = "⚠" if (af["empty"] or af["very_short"] or af["degenerate"]) else " "
        print(f"\n  [{tag} {k}] (tok={af['n_tokens']}, uniq={af['uniq_ratio']}, run={af['max_run']})")
        print("   ", clip(txt[k][i]).replace("\n", " "))
    print()

████████████████████████████████████████████████████████████████████████
DIALOG 478
████████████████████████████████████████████████████████████████████████
             coherence  relevance  fluency  consistency  alignscore
margin_part      0.508      0.911    0.922        0.939       0.594
margin_full      0.142      0.006    0.694        0.757       0.633
finetune         0.936      0.862    0.911        0.944       0.509
[REF]    User Profile:  
Visitor in Cambridge looking for accommodation in the south of the city, interested in swimming, and using taxis for local transport.

Problem:  
Needs a suitable guesthouse in south Cambridge, information on a swimming facility, and a taxi from the hotel to the pool.

Resolution:  
- User initially requested an expensive guesthouse in south Cambridge; none were available, and a north Cambridge option (Avalon) was suggested but not chosen.  
- A moderate-priced guesthouse in south Cambridge, Bridge Guest House, was identified with free park

## 6. Ekspor

Daftar dialog tersangka + verdict otomatis. Setelah inspeksi manual, isi kolom keputusan akhir Anda
(`final_verdict`) dan keluarkan dialog yang terbukti artefak dari analisis berikutnya.

In [6]:
if len(flags):
    flags.to_csv("suspect_rows.csv", index=False)
    if len(low):
        low.to_csv("suspect_lowscore_verdict.csv", index=False)
    # daftar dialog_id unik tersangka untuk di-exclude bila terbukti artefak
    susp_ids = sorted(flags["dialog_id"].unique().tolist())
    pd.Series(susp_ids, name="dialog_id").to_csv("suspect_dialog_ids.csv", index=False)
    print("Disimpan: suspect_rows.csv, suspect_lowscore_verdict.csv, suspect_dialog_ids.csv")
    print(f"Jumlah dialog unik tersangka: {len(susp_ids)} dari {n}")
    print("""
LANGKAH SETELAH INI:
- Baca output sel 5. Jika 'ARTEFAK_*' mendominasi -> dialog itu artefak teknis: keluarkan (exclude)
  dari analisis swinger/ICC, lalu jalankan ulang profiling. Statistik akan jauh lebih bersih.
- Jika banyak 'KEGAGALAN_MODEL?' (output wajar tapi skor rendah) -> itu temuan sungguhan:
  config tsb memang gagal di dialog itu. Lanjut ke analisis pola kegagalan per config.
- Periksa juga apakah satu config (mis. finetune) menyumbang artefak jauh lebih banyak -> bisa jadi
  masalah decoding/generation pada config itu, bukan kualitas model.""")
else:
    print("Tidak ada tersangka pada ambang sekarang. Coba longgarkan LOW_THR / GAP_THR di sel 3.")

Disimpan: suspect_rows.csv, suspect_lowscore_verdict.csv, suspect_dialog_ids.csv
Jumlah dialog unik tersangka: 58 dari 499

LANGKAH SETELAH INI:
- Baca output sel 5. Jika 'ARTEFAK_*' mendominasi -> dialog itu artefak teknis: keluarkan (exclude)
  dari analisis swinger/ICC, lalu jalankan ulang profiling. Statistik akan jauh lebih bersih.
- Jika banyak 'KEGAGALAN_MODEL?' (output wajar tapi skor rendah) -> itu temuan sungguhan:
  config tsb memang gagal di dialog itu. Lanjut ke analisis pola kegagalan per config.
- Periksa juga apakah satu config (mis. finetune) menyumbang artefak jauh lebih banyak -> bisa jadi
  masalah decoding/generation pada config itu, bukan kualitas model.
